<a href="https://colab.research.google.com/github/eodenyire/LearnDataScienceWithEmmanuelOdenyire/blob/main/Phase_1_Foundations_Python_and_Math/Month_02_Data_Analysis_with_Pandas_and_NumPy/Week_3_Pandas_Basics/Day_18_Filtering_and_Querying.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="https://github.com/user-attachments/assets/2a4adf95-3d28-41e2-90d0-d06b2e9c8fa3" alt="Learn Data Science with Emmanuel Odenyire" width="180"/>

# Day 18: Filtering and Querying DataFrames

## Phase 1 | Month 2 | Week 3 | Day 18

**Goal:** Master advanced filtering techniques — combining conditions, `isin()`, `between()`, regex filtering, and performance tips.

### What You Will Learn
1. Combining boolean masks with `&`, `|`, `~`
2. `isin()`, `between()`, `str.contains()`
3. Filtering with functions: `.apply(lambda ...)`
4. Duplicate detection and removal
5. Performance comparison: mask vs query

### Resources
- 📖 [Pandas Filtering Guide](https://pandas.pydata.org/docs/user_guide/indexing.html#boolean-indexing)
- 🎥 [Pandas Filtering — Corey Schafer](https://www.youtube.com/watch?v=Lw2rlcxScZY)

In [ ]:
import pandas as pd
import numpy as np
np.random.seed(42)

# Create a rich government employee dataset
n = 500
df = pd.DataFrame({
    'emp_id'    : range(1, n+1),
    'county'    : np.random.choice(['Nairobi','Mombasa','Kisumu','Nakuru','Eldoret','Thika','Kisii','Meru'], n),
    'department': np.random.choice(['IT','Health','Education','Finance','Infrastructure'], n),
    'grade'     : np.random.choice(['A1','A2','B1','B2','C1','C2'], n),
    'salary'    : np.random.randint(40_000, 400_000, n),
    'age'       : np.random.randint(22, 60, n),
    'years_service': np.random.randint(0, 35, n),
})

print('Government employees dataset:', df.shape)
print(df.head())


In [ ]:
import pandas as pd
import numpy as np
np.random.seed(42)
n = 500
df = pd.DataFrame({
    'emp_id': range(1,n+1),
    'county': np.random.choice(['Nairobi','Mombasa','Kisumu','Nakuru','Eldoret','Thika','Kisii','Meru'],n),
    'department': np.random.choice(['IT','Health','Education','Finance','Infrastructure'],n),
    'grade': np.random.choice(['A1','A2','B1','B2','C1','C2'],n),
    'salary': np.random.randint(40_000,400_000,n),
    'age': np.random.randint(22,60,n),
    'years_service': np.random.randint(0,35,n),
})

# isin() — match any of a list
big_cities = ['Nairobi', 'Mombasa', 'Kisumu']
print('Major city employees:', df[df['county'].isin(big_cities)].shape[0])

# between() — range filter
mid_salary = df[df['salary'].between(80_000, 200_000)]
print(f'Mid-range salary ({80_000}-{200_000}): {len(mid_salary)} employees')

# Multiple conditions
senior_it = df[(df['department'] == 'IT') &
               (df['years_service'] >= 10) &
               (df['grade'].isin(['A1', 'A2']))]
print(f'Senior IT (A1/A2, 10+ yrs): {len(senior_it)}')

# NOT operator
not_nairobi = df[~df['county'].str.contains('Nairobi')]
print(f'Outside Nairobi: {len(not_nairobi)}')

# Combine with .query()
high_earners = df.query('salary > 300_000 and department in ["IT", "Finance"]')
print(f'\nHigh earners (IT/Finance >300k): {len(high_earners)}')
print(high_earners[['county','department','grade','salary']].head())

In [ ]:
import pandas as pd
import numpy as np
np.random.seed(42)
n = 500
df = pd.DataFrame({
    'emp_id': range(1,n+1),
    'county': np.random.choice(['Nairobi','Mombasa','Kisumu'],n),
    'department': np.random.choice(['IT','Health','Education'],n),
    'salary': np.random.randint(40_000,400_000,n),
    'age': np.random.randint(22,60,n),
})

# .apply() for complex custom conditions
def categorise_salary(row):
    if row['salary'] > 300_000:
        return 'Executive'
    elif row['salary'] > 150_000:
        return 'Senior'
    elif row['salary'] > 80_000:
        return 'Mid'
    else:
        return 'Junior'

df['level'] = df.apply(categorise_salary, axis=1)
print('Salary levels:')
print(df['level'].value_counts())

# Faster: np.select or pd.cut
df['level_fast'] = pd.cut(df['salary'], bins=[0,80000,150000,300000,float('inf')],
                          labels=['Junior','Mid','Senior','Executive'])
print('\nMatch:', (df['level'] == df['level_fast'].astype(str)).all())

# Duplicates
df_with_dupes = pd.concat([df.head(20), df.head(5)])  # add 5 duplicates
print(f'\nBefore dedup: {len(df_with_dupes)} rows')
print(f'Duplicates  : {df_with_dupes.duplicated().sum()}')
df_clean = df_with_dupes.drop_duplicates()
print(f'After dedup : {len(df_clean)} rows')

## Exercises

### Exercise 1: KCPE Results Analysis
Filter and analyse national exam results.

In [ ]:
import pandas as pd
import numpy as np
np.random.seed(2024)

n = 2000
df = pd.DataFrame({
    'student_id': range(1, n+1),
    'gender'    : np.random.choice(['M','F'], n),
    'county'    : np.random.choice(['Nairobi','Kisumu','Mombasa','Nakuru','Eldoret'], n),
    'school_type': np.random.choice(['Public','Private','Mission'], n),
    'maths'     : np.random.randint(0, 100, n),
    'english'   : np.random.randint(0, 100, n),
    'science'   : np.random.randint(0, 100, n),
    'swahili'   : np.random.randint(0, 100, n),
    'social'    : np.random.randint(0, 100, n),
})
df['total'] = df[['maths','english','science','swahili','social']].sum(axis=1)
df['grade']  = pd.cut(df['total'], bins=[0,150,200,250,350,500],
                      labels=['E','D','C','B','A'])

# 1. How many students scored A?
# 2. Top performing county (mean total)
# 3. Performance gap: private vs public schools
# 4. Girls vs boys average in maths
# 5. Students who scored top 10% in all 5 subjects
# YOUR CODE HERE

## Summary

- `isin()` replaces multiple `==` conditions
- `between()` is a cleaner alternative to `& >=` and `<= &`
- `.apply(func, axis=1)` for row-wise custom logic (use sparingly — slow)
- `pd.cut()` is the fast vectorised way to bin numeric data

**Next: Day 19 — Handling Missing Data**